In [1]:
import pandas as pd
import numpy as np
import duckdb
import os

In [2]:
con = duckdb.connect()

con.execute("SET preserve_insertion_order=false")#dont keep row order

#Create a reference to the bus load files
con.execute("""
CREATE OR REPLACE VIEW bus_load AS
SELECT * FROM 'bus_load_2022.parquet'
UNION ALL
SELECT * FROM 'bus_load_2023.parquet'
UNION ALL
SELECT * FROM 'bus_load_2024.parquet'
UNION ALL
SELECT * FROM 'bus_load_2025.parquet'
""")

In [3]:
#extract features from all bus_load parquet files to a temporary working parquet file
con.execute("""
COPY (
SELECT
    bus_unique_id,
    zone_name,
    CAST(date AS DATE) AS date,
    he,
    EXTRACT(DOW FROM CAST(date AS DATE)) AS dow,
    EXTRACT(DOY FROM CAST(date AS DATE)) AS day_of_year,
    EXTRACT(MONTH FROM CAST(date AS DATE)) AS month,
    CAST(pd AS FLOAT) AS pd,
    CAST(date AS TIMESTAMP)
        + INTERVAL (he - 1) HOUR
        AS ts,
    CASE
        WHEN dow IN (0, 6)
        THEN 1
        ELSE 0
    END AS is_weekend
FROM bus_load
WHERE pd IS NOT NULL
)

TO 'working.parquet'
(FORMAT PARQUET)

""")

In [4]:
# Compute expensive lag features and save to a temporary lagged_features parquet
con.execute("""
COPY (
SELECT
    a.*,
    CAST(b.pd AS FLOAT) AS same_hour_prev_week,
    CAST(c.pd AS FLOAT) AS same_hour_prev_35day,
    CAST(d.pd AS FLOAT) AS same_hour_prev_year
FROM 'working.parquet' a

LEFT JOIN (SELECT bus_unique_id, date, he, pd FROM 'working.parquet') AS b
    ON a.bus_unique_id = b.bus_unique_id
   AND a.date = b.date + INTERVAL 7 DAY
   AND a.he = b.he

LEFT JOIN (SELECT bus_unique_id, date, he, pd FROM 'working.parquet') AS c
    ON a.bus_unique_id = c.bus_unique_id
   AND a.date = c.date + INTERVAL 35 DAY
   AND a.he = c.he

LEFT JOIN (SELECT bus_unique_id, date, he, pd FROM 'working.parquet') AS d
    ON a.bus_unique_id = d.bus_unique_id
   AND a.date = d.date + INTERVAL 1 YEAR
   AND a.he = d.he
)

TO 'features.parquet'
(FORMAT PARQUET)

""")
os.remove('working.parquet') #remove the temporary working parquet from the directory


In [5]:
#Create a reference to the zone load files
con.execute("""
CREATE OR REPLACE VIEW zone_load AS
SELECT * FROM 'zone_load_2022.parquet'
UNION ALL
SELECT * FROM 'zone_load_2023.parquet'
UNION ALL
SELECT * FROM 'zone_load_2024.parquet'
UNION ALL
SELECT * FROM 'zone_load_2025.parquet'
""")

In [6]:
#extract features from all bus_load parquet files to a temporary working parquet file
con.execute("""
COPY (
SELECT
    zone_name,
    CAST(date AS DATE) AS date,
    he,
    EXTRACT(DOW FROM CAST(date AS DATE)) AS dow,
    EXTRACT(DOY FROM CAST(date AS DATE)) AS day_of_year,
    EXTRACT(MONTH FROM CAST(date AS DATE)) AS month,
    CAST(pd AS FLOAT) AS pd,
    CAST(date AS TIMESTAMP)
        + INTERVAL (he - 1) HOUR
        AS ts,
    CASE
        WHEN dow IN (0, 6)
        THEN 1
        ELSE 0
    END AS is_weekend
    
FROM zone_load
WHERE pd IS NOT NULL
)

TO 'zone_working.parquet'
(FORMAT PARQUET)

""")

In [7]:
con.execute("""

COPY (
SELECT
    a.*,
    CAST(b.pd AS FLOAT) AS same_hour_prev_week,
    CAST(c.pd AS FLOAT) AS same_hour_prev_35day,
    CAST(d.pd AS FLOAT) AS same_hour_prev_year
FROM 'zone_working.parquet' AS a

LEFT JOIN (SELECT zone_name, date, he, pd FROM 'zone_working.parquet') AS b
    ON a.zone_name = b.zone_name
   AND a.date = b.date + INTERVAL 7 DAY
   AND a.he = b.he

LEFT JOIN (SELECT zone_name, date, he, pd FROM 'zone_working.parquet') AS c
    ON a.zone_name = c.zone_name
   AND a.date = c.date + INTERVAL 35 DAY
   AND a.he = c.he

LEFT JOIN (SELECT zone_name, date, he, pd FROM 'zone_working.parquet') AS d
    ON a.zone_name = d.zone_name
   AND a.date = d.date + INTERVAL 1 YEAR
   AND a.he = d.he
)

TO 'zone_features.parquet'
(FORMAT PARQUET)

""")
os.remove('zone_working.parquet')

In [8]:
#get 2 month lagged bus share and save as lagged_bus_share
con.execute("""
COPY (
WITH base AS (
    SELECT
        bus_unique_id,
        zone_name,
        CAST(date AS DATE) AS date,
        EXTRACT(YEAR FROM date) AS year,
        EXTRACT(MONTH FROM date) AS month,
        he,
        pd
    FROM 'features.parquet'
),

zone_hourly AS (
    SELECT
        zone_name,
        date,
        he,
        SUM(pd) AS zone_pd
    FROM base
    GROUP BY
        zone_name,
        date,
        he
),

hourly_share AS (
    SELECT
        b.bus_unique_id,
        b.zone_name,
        b.year,
        b.month,
        b.he,
        b.pd / z.zone_pd AS bus_share
    FROM base b
    JOIN zone_hourly z
        ON b.zone_name = z.zone_name
       AND b.date = z.date
       AND b.he = z.he
    WHERE z.zone_pd != 0
),

monthly_hourly_share AS (
    SELECT
        bus_unique_id,
        zone_name,
        year,
        month,
        he,
        AVG(bus_share) AS avg_bus_share
    FROM hourly_share
    GROUP BY
        bus_unique_id,
        zone_name,
        year,
        month,
        he
),

lagged_share AS (
    SELECT
        CASE
            WHEN month >= 11 THEN year + 1
            ELSE year
        END AS target_year,
        CASE
            WHEN month = 11 THEN 1
            WHEN month = 12 THEN 2
            ELSE month + 2
        END AS target_month,
        bus_unique_id AS bus_id,
        zone_name,
        he,
        avg_bus_share
    FROM monthly_hourly_share
)
SELECT *
FROM lagged_share
ORDER BY
    target_year,
    target_month,
    zone_name,
    he,
    bus_id
)
TO 'lagged_bus_share.parquet'
(FORMAT PARQUET)

""")